# 🕸️ Notebook 02 — Food Knowledge Graph

This notebook builds and explores the **Food Knowledge Graph** that powers candidate generation in SmartCart AI.

## Graph Structure
- **133 nodes**: Items, Categories, Cuisines, Time Contexts
- **393 edges**: `pairs_with`, `belongs_to`, `best_time`, `goes_with_weather`

The graph enables SmartCart to retrieve semantically relevant add-on candidates based on what's in the cart.

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from collections import Counter, defaultdict

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

DATA_RAW = os.path.join(os.getcwd(), '..', 'data', 'raw')
print('Loaded libraries. NetworkX version:', nx.__version__)

In [ ]:
# Load raw data
with open(os.path.join(DATA_RAW, 'menu_items.json')) as f:
    menu_data = json.load(f)
with open(os.path.join(DATA_RAW, 'food_pairings.json')) as f:
    pairings_data = json.load(f)

if isinstance(menu_data, list):
    items = menu_data
elif isinstance(menu_data, dict) and 'items' in menu_data:
    items = menu_data['items']
else:
    items = list(menu_data.values())

if isinstance(pairings_data, list):
    pairings = pairings_data
elif isinstance(pairings_data, dict) and 'pairings' in pairings_data:
    pairings = pairings_data['pairings']
else:
    pairings = list(pairings_data.values())

print(f'Items: {len(items)}, Pairings: {len(pairings)}')

In [ ]:
# Build the Food Knowledge Graph
G = nx.DiGraph()

name_field = next((k for k in ['name', 'item_name', 'food_name', 'id'] if k in items[0]), list(items[0].keys())[0])
cat_field  = next((k for k in ['category', 'type', 'food_type'] if k in items[0]), None)
cuisine_field = next((k for k in ['cuisine', 'cuisine_type'] if k in items[0]), None)

# Add item nodes
for item in items:
    node_id = item[name_field]
    attrs = {'node_type': 'item'}
    for key in ['price', 'price_inr', 'kpt_minutes', 'kpt', 'margin', 'category', 'cuisine', 'is_veg']:
        if key in item:
            attrs[key] = item[key]
    G.add_node(node_id, **attrs)

# Add category and cuisine nodes
if cat_field:
    for item in items:
        cat = item.get(cat_field)
        if cat:
            if not G.has_node(cat):
                G.add_node(cat, node_type='category')
            G.add_edge(item[name_field], cat, edge_type='belongs_to')

if cuisine_field:
    for item in items:
        cuisine = item.get(cuisine_field)
        if cuisine:
            if not G.has_node(cuisine):
                G.add_node(cuisine, node_type='cuisine')
            G.add_edge(item[name_field], cuisine, edge_type='cuisine_of')

# Add pairing edges
pair_keys = list(pairings[0].keys())
src_key = next((k for k in ['item1', 'food1', 'source', 'from', 'base_item'] if k in pair_keys), pair_keys[0])
tgt_key = next((k for k in ['item2', 'food2', 'target', 'to', 'addon_item'] if k in pair_keys), pair_keys[1])
str_key = next((k for k in ['strength', 'pairing_strength', 'score', 'weight'] if k in pair_keys), None)

for pair in pairings:
    src, tgt = str(pair[src_key]), str(pair[tgt_key])
    weight = float(pair[str_key]) if str_key else 0.5
    G.add_edge(src, tgt, edge_type='pairs_with', weight=weight)

print(f'\n📊 Graph built:')
print(f'   Nodes: {G.number_of_nodes()}')
print(f'   Edges: {G.number_of_edges()}')

node_types = Counter(nx.get_node_attributes(G, 'node_type').values())
edge_types = Counter(nx.get_edge_attributes(G, 'edge_type').values())
print(f'   Node types: {dict(node_types)}')
print(f'   Edge types: {dict(edge_types)}')

In [ ]:
# Graph statistics
item_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'item']
item_subgraph = G.subgraph(item_nodes)

in_degrees  = dict(G.in_degree(item_nodes))
out_degrees = dict(G.out_degree(item_nodes))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# In-degree (how often item appears as add-on)
in_vals = list(in_degrees.values())
axes[0].hist(in_vals, bins=15, color='#3498DB', edgecolor='white')
axes[0].set_title('In-Degree Distribution\n(how often item is suggested as add-on)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('In-Degree')
axes[0].set_ylabel('Number of Items')

# Top 10 most suggested add-ons
top_addons = sorted(in_degrees.items(), key=lambda x: x[1], reverse=True)[:10]
names, counts = zip(*top_addons) if top_addons else ([], [])
axes[1].barh(list(names), list(counts), color='#E74C3C')
axes[1].set_title('Top 10 Most Paired Add-ons', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Items it Pairs With')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Sample query: Get add-on candidates for 'Chicken Biryani'
def get_candidates(G, cart_item, top_n=10):
    """Get add-on candidates from knowledge graph for a cart item."""
    candidates = []
    if cart_item not in G:
        # Try partial match
        matches = [n for n in G.nodes() if cart_item.lower() in str(n).lower()]
        if matches:
            cart_item = matches[0]
            print(f'  Matched to: {cart_item}')
        else:
            return []

    for successor in G.successors(cart_item):
        edge_data = G.edges[cart_item, successor]
        if edge_data.get('edge_type') == 'pairs_with':
            node_data = G.nodes[successor]
            candidates.append({
                'name': successor,
                'strength': edge_data.get('weight', 0.5),
                'category': node_data.get('category', 'Unknown'),
                'price': node_data.get('price', node_data.get('price_inr', '?')),
                'kpt': node_data.get('kpt_minutes', node_data.get('kpt', '?'))
            })

    return sorted(candidates, key=lambda x: x['strength'], reverse=True)[:top_n]

print('🍛 Add-on candidates for Chicken Biryani:')
print('-' * 55)
biryani_candidates = get_candidates(G, 'Chicken Biryani')
if biryani_candidates:
    for i, c in enumerate(biryani_candidates, 1):
        print(f"  {i}. {c['name']:<25} strength={c['strength']:.2f}  cat={c['category']}")
else:
    print('  [No direct pairings found — check item name spelling in food_pairings.json]')

print()
print('🍕 Add-on candidates for Margherita Pizza:')
print('-' * 55)
pizza_candidates = get_candidates(G, 'Margherita Pizza')
if pizza_candidates:
    for i, c in enumerate(pizza_candidates, 1):
        print(f"  {i}. {c['name']:<25} strength={c['strength']:.2f}  cat={c['category']}")
else:
    print('  [No direct pairings found — check item name spelling in food_pairings.json]')

In [ ]:
# Visualize a small sub-graph around a popular item
focus_item = None
for candidate in ['Chicken Biryani', 'Margherita Pizza', 'Veg Burger', items[0][name_field]]:
    if candidate in G:
        focus_item = candidate
        break

if focus_item:
    # Build ego graph (1-hop neighborhood)
    neighbors = list(G.successors(focus_item)) + list(G.predecessors(focus_item))
    sub_nodes = [focus_item] + neighbors[:14]  # limit for readability
    sub_G = G.subgraph(sub_nodes)

    fig, ax = plt.subplots(figsize=(12, 8))
    pos = nx.spring_layout(sub_G, seed=42, k=2)

    node_colors = []
    for n in sub_G.nodes():
        nt = sub_G.nodes[n].get('node_type', 'item')
        if n == focus_item:
            node_colors.append('#E74C3C')
        elif nt == 'category':
            node_colors.append('#F39C12')
        elif nt == 'cuisine':
            node_colors.append('#27AE60')
        else:
            node_colors.append('#3498DB')

    nx.draw_networkx(sub_G, pos=pos, ax=ax,
                     node_color=node_colors,
                     node_size=1200,
                     font_size=7,
                     arrows=True,
                     edge_color='#BDC3C7',
                     width=1.5)

    from matplotlib.patches import Patch
    legend = [
        Patch(color='#E74C3C', label='Focus Item'),
        Patch(color='#3498DB', label='Food Item'),
        Patch(color='#F39C12', label='Category'),
        Patch(color='#27AE60', label='Cuisine'),
    ]
    ax.legend(handles=legend, loc='upper left')
    ax.set_title(f'Knowledge Graph Neighborhood: {focus_item}', fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Could not find a focus item in the graph')

In [ ]:
# Graph summary
print('=' * 50)
print('KNOWLEDGE GRAPH SUMMARY')
print('=' * 50)
print(f'Total nodes:         {G.number_of_nodes()}')
print(f'Total edges:         {G.number_of_edges()}')
print(f'Node type breakdown: {dict(node_types)}')
print(f'Edge type breakdown: {dict(edge_types)}')
print(f'Graph density:       {nx.density(G):.4f}')
try:
    ug = G.to_undirected()
    comps = list(nx.connected_components(ug))
    print(f'Connected components: {len(comps)}')
    print(f'Largest component:    {max(len(c) for c in comps)} nodes')
except Exception as e:
    print(f'Component analysis: {e}')